## Imports

In [9]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [8]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "spotify_churn_dataset.csv"
CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "spotify_churn_cleaned.csv"
MODEL_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "spotify_churn_model_matrix.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)

Project root: c:\Users\coolc\OneDrive\Desktop\spotify-churn-analysis\spotify-churn-analysis
Raw data path: c:\Users\coolc\OneDrive\Desktop\spotify-churn-analysis\spotify-churn-analysis\data\raw\spotify_churn_dataset.csv


## Raw data loading

In [3]:
df_raw = pd.read_csv(RAW_DATA_PATH)

df_raw.head()
df_raw.shape

(8000, 12)

## Basic audit of the data

In [21]:
df_raw.info()

display(df_raw.describe(include="all"))

missing_values = df_raw.isna().sum()
display(missing_values)

duplicate_rows = df_raw.duplicated().sum()
duplicate_user_ids = df_raw["user_id"].duplicated().sum()

print("Duplicate rows:", duplicate_rows)
print("Duplicate user IDs:", duplicate_user_ids)

display(df_raw.nunique().sort_values())

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 12 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                8000 non-null   int64  
 1   gender                 8000 non-null   str    
 2   age                    8000 non-null   int64  
 3   country                8000 non-null   str    
 4   subscription_type      8000 non-null   str    
 5   listening_time         8000 non-null   int64  
 6   songs_played_per_day   8000 non-null   int64  
 7   skip_rate              8000 non-null   float64
 8   device_type            8000 non-null   str    
 9   ads_listened_per_week  8000 non-null   int64  
 10  offline_listening      8000 non-null   int64  
 11  is_churned             8000 non-null   int64  
dtypes: float64(1), int64(7), str(4)
memory usage: 750.1 KB


,user_id,gender,age,country,subscription_type,listening_time,songs_played_per_day,skip_rate,device_type,ads_listened_per_week,offline_listening,is_churned
count,8000.00000,8000,8000.000000,8000,8000,8000.000000,8000.000000,8000.000000,8000,8000.000000,8000.000000,8000.000000
unique,NaN,3,NaN,8,4,NaN,NaN,NaN,3,NaN,NaN,NaN
top,NaN,Male,NaN,AU,Premium,NaN,NaN,NaN,Desktop,NaN,NaN,NaN
freq,NaN,2691,NaN,1034,2115,NaN,NaN,NaN,2778,NaN,NaN,NaN
mean,4000.50000,NaN,37.662125,NaN,NaN,154.068250,50.127250,0.300127,NaN,6.943875,0.747750,0.258875
std,2309.54541,NaN,12.740359,NaN,NaN,84.015596,28.449762,0.173594,NaN,13.617953,0.434331,0.438044
min,1.00000,NaN,16.000000,NaN,NaN,10.000000,1.000000,0.000000,NaN,0.000000,0.000000,0.000000
25%,2000.75000,NaN,26.000000,NaN,NaN,81.000000,25.000000,0.150000,NaN,0.000000,0.000000,0.000000
50%,4000.50000,NaN,38.000000,NaN,NaN,154.000000,50.000000,0.300000,NaN,0.000000,1.000000,0.000000
75%,6000.25000,NaN,49.000000,NaN,NaN,227.000000,75.000000,0.450000,NaN,5.000000,1.000000,1.000000


user_id                  0
gender                   0
age                      0
country                  0
subscription_type        0
listening_time           0
songs_played_per_day     0
skip_rate                0
device_type              0
ads_listened_per_week    0
offline_listening        0
is_churned               0
dtype: int64

Duplicate rows: 0
Duplicate user IDs: 0


offline_listening           2
is_churned                  2
gender                      3
device_type                 3
subscription_type           4
country                     8
age                        44
ads_listened_per_week      46
skip_rate                  61
songs_played_per_day       99
listening_time            290
user_id                  8000
dtype: int64

## Validation

Because the dataset is already pretty well-structured, the cleaning stage focuses on validation, minor standardization, and preparing a ready version for clustering.

In [11]:
df = df_raw.copy()

In [12]:
validation_checks = {
    "missing_values": df.isna().sum().sum(),
    "duplicate_rows": df.duplicated().sum(),
    "duplicate_user_ids": df["user_id"].duplicated().sum(),
    "age_below_0": (df["age"] < 0).sum(),
    "listening_time_below_0": (df["listening_time"] < 0).sum(),
    "songs_played_per_day_below_0": (df["songs_played_per_day"] < 0).sum(),
    "skip_rate_below_0": (df["skip_rate"] < 0).sum(),
    "skip_rate_above_1": (df["skip_rate"] > 1).sum(),
    "ads_listened_per_week_below_0": (df["ads_listened_per_week"] < 0).sum(),
    "offline_listening_invalid": (~df["offline_listening"].isin([0, 1])).sum(),
    "is_churned_invalid": (~df["is_churned"].isin([0, 1])).sum(),
}

validation_results = pd.Series(validation_checks)
validation_results

missing_values                   0
duplicate_rows                   0
duplicate_user_ids               0
age_below_0                      0
listening_time_below_0           0
songs_played_per_day_below_0     0
skip_rate_below_0                0
skip_rate_above_1                0
ads_listened_per_week_below_0    0
offline_listening_invalid        0
is_churned_invalid               0
dtype: int64

## Validation Results

All validation checks returned 0. This means the dataset has no missing values, no duplicate rows, no duplicate user IDs, no invalid binary labels, and no invalid numeric ranges for the fields checked.

Because the raw data is already clean, no rows needed to be removed and no missing values needed to be imputed. The remaining preprocessing steps focus on standardizing categorical text, excluding non-modeling columns, encoding categorical variables, and scaling numeric features for clustering.

## Standardization

In [13]:
df_clean = df.copy()

categorical_cols = [
    "gender",
    "country",
    "subscription_type",
    "device_type"
]

for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

df_clean["gender"] = df_clean["gender"].str.title()
df_clean["country"] = df_clean["country"].str.upper()
df_clean["subscription_type"] = df_clean["subscription_type"].str.title()
df_clean["device_type"] = df_clean["device_type"].str.title()

for col in categorical_cols:
    print(f"\n{col}")
    print(sorted(df_clean[col].unique()))


gender
['Female', 'Male', 'Other']

country
['AU', 'CA', 'DE', 'FR', 'IN', 'PK', 'UK', 'US']

subscription_type
['Family', 'Free', 'Premium', 'Student']

device_type
['Desktop', 'Mobile', 'Web']


In [14]:
id_col = "user_id"
target_col = "is_churned"

numeric_cols = [
    "age",
    "listening_time",
    "songs_played_per_day",
    "skip_rate",
    "ads_listened_per_week"
]

binary_cols = [
    "offline_listening"
]

cluster_feature_cols = numeric_cols + binary_cols + categorical_cols

## Modeling Column Decisions

The `user_id` column was excluded because it is only an identifier and does not describe user behavior. The `is_churned` column was also excluded from clustering to avoid target leakage. It will be used later to evaluate whether the discovered clusters have different churn rates. When clustering, they can be dropped, but we add them back later for analysis after clustering.

In [15]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(CLEAN_DATA_PATH, index=False)

print(f"Saved cleaned data to: {CLEAN_DATA_PATH}")

Saved cleaned data to: c:\Users\coolc\OneDrive\Desktop\spotify-churn-analysis\spotify-churn-analysis\data\processed\spotify_churn_cleaned.csv


In [16]:
X = df_clean[cluster_feature_cols].copy()
y = df_clean[target_col].copy()

X.head()

,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,gender,country,subscription_type,device_type
0,54,26,23,0.20,31,0,Female,CA,Free,Desktop
1,33,141,62,0.34,0,1,Other,DE,Family,Web
2,38,199,38,0.04,0,1,Male,AU,Premium,Mobile
3,22,36,2,0.31,0,1,Female,CA,Student,Mobile
4,29,250,57,0.36,0,1,Other,US,Family,Mobile


In [17]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("bin", "passthrough", binary_cols),
        ("cat", make_one_hot_encoder(), categorical_cols),
    ],
    remainder="drop"
)

X_processed = preprocessor.fit_transform(X)

X_processed.shape

(8000, 24)

In [18]:
cat_feature_names = (
    preprocessor
    .named_transformers_["cat"]
    .get_feature_names_out(categorical_cols)
    .tolist()
)

processed_feature_names = numeric_cols + binary_cols + cat_feature_names

X_processed_df = pd.DataFrame(X_processed, columns=processed_feature_names)

X_processed_df.head()

,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,gender_Female,gender_Male,gender_Other,country_AU,...,country_PK,country_UK,country_US,subscription_type_Family,subscription_type_Free,subscription_type_Premium,subscription_type_Student,device_type_Desktop,device_type_Mobile,device_type_Web
0,1.282452,-1.524434,-0.953574,-0.576827,1.766611,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
1,-0.365956,-0.155555,0.417349,0.229702,-0.509938,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.026522,0.534836,-0.426296,-1.498575,-0.509938,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3,-1.229408,-1.405401,-1.691763,0.056875,-0.509938,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,-0.679939,1.141904,0.241590,0.344921,-0.509938,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [19]:
model_df = X_processed_df.copy()
model_df[id_col] = df_clean[id_col].values
model_df[target_col] = df_clean[target_col].values

model_df.head()

,age,listening_time,songs_played_per_day,skip_rate,ads_listened_per_week,offline_listening,gender_Female,gender_Male,gender_Other,country_AU,...,country_US,subscription_type_Family,subscription_type_Free,subscription_type_Premium,subscription_type_Student,device_type_Desktop,device_type_Mobile,device_type_Web,user_id,is_churned
0,1.282452,-1.524434,-0.953574,-0.576827,1.766611,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1,1
1,-0.365956,-0.155555,0.417349,0.229702,-0.509938,1.0,0.0,0.0,1.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,2,0
2,0.026522,0.534836,-0.426296,-1.498575,-0.509938,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,3,1
3,-1.229408,-1.405401,-1.691763,0.056875,-0.509938,1.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,4,0
4,-0.679939,1.141904,0.241590,0.344921,-0.509938,1.0,0.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,5,1


In [20]:
model_df.to_csv(MODEL_DATA_PATH, index=False)

print(f"Saved model-ready data to: {MODEL_DATA_PATH}")

Saved model-ready data to: c:\Users\coolc\OneDrive\Desktop\spotify-churn-analysis\spotify-churn-analysis\data\processed\spotify_churn_model_matrix.csv
